# 🏠 House Price Predictor
**Author:** [Your Name]  
**Dataset:** [Kaggle House Prices - Advanced Regression Techniques](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data)  

## Project Overview
This project builds a machine learning model to predict residential home sale prices in Ames, Iowa.
Using 79 features describing nearly every aspect of a home, we explore what drives value,
engineer meaningful features, and compare multiple regression models.

**Business Question:** *Which features most significantly drive home prices, and how accurately can we predict sale price from property characteristics?*

---
## Table of Contents
1. Setup & Data Loading
2. Exploratory Data Analysis (EDA)
3. Data Cleaning & Preprocessing
4. Feature Engineering
5. Model Building & Evaluation
6. Feature Importance & Business Insights
7. Conclusions

## 1. Setup & Data Loading

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine Learning ─────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# ── Plot styling ─────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print('✅ All libraries loaded successfully!')

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
# Download from Kaggle: https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/data
# Place train.csv and test.csv in the same folder as this notebook

df = pd.read_csv('train.csv')

print(f'Dataset shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

In [ ]:
# Quick summary of the data
df.info()

In [ ]:
# Basic statistics for numerical columns
df.describe().T.style.background_gradient(cmap='Blues')

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# ── Target variable: SalePrice distribution ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw distribution
axes[0].hist(df['SalePrice'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Sale Price Distribution (Raw)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Count')

# Log-transformed (helps ML models)
axes[1].hist(np.log1p(df['SalePrice']), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Sale Price Distribution (Log-Transformed)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Log(Sale Price)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('saleprice_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Mean Sale Price:   ${df['SalePrice'].mean():,.0f}")
print(f"Median Sale Price: ${df['SalePrice'].median():,.0f}")
print(f"Min Sale Price:    ${df['SalePrice'].min():,.0f}")
print(f"Max Sale Price:    ${df['SalePrice'].max():,.0f}")

In [ ]:
# ── Correlation heatmap: top numerical features ───────────────────────────────
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_matrix = df[numeric_cols].corr()

# Top 15 features most correlated with SalePrice
top_corr = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(15).index

plt.figure(figsize=(12, 10))
sns.heatmap(
    df[top_corr].corr(),
    annot=True, fmt='.2f',
    cmap='RdYlGn', center=0,
    square=True, linewidths=0.5
)
plt.title('Correlation Matrix: Top 15 Features vs Sale Price', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Key scatter plots: size vs price ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

scatter_features = [
    ('GrLivArea',    'Above Ground Living Area (sq ft)'),
    ('TotalBsmtSF',  'Total Basement Area (sq ft)'),
    ('GarageArea',   'Garage Area (sq ft)'),
    ('OverallQual',  'Overall Quality Rating (1-10)'),
]

for ax, (feature, label) in zip(axes.flatten(), scatter_features):
    ax.scatter(df[feature], df['SalePrice'], alpha=0.4, color='steelblue', s=20)
    ax.set_xlabel(label)
    ax.set_ylabel('Sale Price ($)')
    ax.set_title(f'{label} vs Sale Price', fontweight='bold')

plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sale Price by Neighborhood (business insight!) ───────────────────────────
neighborhood_price = df.groupby('Neighborhood')['SalePrice'].median().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
neighborhood_price.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Median Sale Price by Neighborhood', fontsize=14, fontweight='bold')
plt.xlabel('Neighborhood')
plt.ylabel('Median Sale Price ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('neighborhood_prices.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Data Cleaning & Preprocessing

In [ ]:
# ── Check missing values ──────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(f'Columns with missing values: {len(missing_df)}')
missing_df.head(20)

In [ ]:
# ── Drop columns with >40% missing data ──────────────────────────────────────
high_missing = missing_pct[missing_pct > 40].index.tolist()
print(f'Dropping columns (>40% missing): {high_missing}')
df.drop(columns=high_missing, inplace=True)

# ── Fill remaining missing values ────────────────────────────────────────────
# For many categorical NaN values, 'None' means the feature simply doesn't exist
cat_fill_none = ['GarageType','GarageFinish','GarageQual','GarageCond',
                  'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
                  'MasVnrType','FireplaceQu']

for col in cat_fill_none:
    if col in df.columns:
        df[col].fillna('None', inplace=True)

# Numerical NaN → fill with median
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Remaining categorical → fill with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Missing values remaining: {df.isnull().sum().sum()}')

In [ ]:
# ── Remove extreme outliers in GrLivArea ─────────────────────────────────────
# Two data points are clearly anomalous (very large houses sold very cheap)
before = len(df)
df = df[~((df['GrLivArea'] > 4000) & (df['SalePrice'] < 200000))]
print(f'Removed {before - len(df)} outlier rows. Dataset now has {len(df):,} rows.')

## 4. Feature Engineering

In [ ]:
# ── Create new meaningful features ───────────────────────────────────────────

# Total square footage
df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']

# Total bathrooms (full bath counts more than half bath)
df['TotalBathrooms'] = (
    df['FullBath'] + (0.5 * df['HalfBath']) +
    df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
)

# Age of house at time of sale
df['HouseAge'] = df['YrSold'] - df['YearBuilt']

# Years since last remodel
df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']

# Was the house remodeled?
df['WasRemodeled'] = (df['YearRemodAdd'] != df['YearBuilt']).astype(int)

# Has a garage?
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)

# Has a basement?
df['HasBasement'] = (df['TotalBsmtSF'] > 0).astype(int)

print('✅ New features created:')
new_features = ['TotalSF', 'TotalBathrooms', 'HouseAge', 'YearsSinceRemodel',
                'WasRemodeled', 'HasGarage', 'HasBasement']
print(df[new_features].describe().T[['mean','min','max']])

In [ ]:
# ── Encode categorical variables ──────────────────────────────────────────────
# We'll use one-hot encoding for nominal categories

# Separate target variable
y = np.log1p(df['SalePrice'])  # log-transform makes distribution more normal
X = df.drop(columns=['SalePrice', 'Id'])

# One-hot encode all remaining categorical columns
X = pd.get_dummies(X, drop_first=True)

print(f'Feature matrix shape: {X.shape}')
print(f'Target variable shape: {y.shape}')

## 5. Model Building & Evaluation

In [ ]:
# ── Train/test split ──────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (important for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Training set:   {X_train.shape[0]:,} samples')
print(f'Test set:       {X_test.shape[0]:,} samples')

In [ ]:
# ── Train multiple models and compare ────────────────────────────────────────

models = {
    'Linear Regression':        LinearRegression(),
    'Ridge Regression':         Ridge(alpha=10),
    'Lasso Regression':         Lasso(alpha=0.001),
    'Random Forest':            RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':        GradientBoostingRegressor(n_estimators=200, random_state=42),
}

results = []

for name, model in models.items():
    # Linear models use scaled features; tree models don't need scaling
    if 'Regression' in name:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Convert predictions back from log scale
    y_pred_actual = np.expm1(y_pred)
    y_test_actual = np.expm1(y_test)
    
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    mae  = mean_absolute_error(y_test_actual, y_pred_actual)
    r2   = r2_score(y_test_actual, y_pred_actual)
    
    results.append({'Model': name, 'RMSE ($)': f'{rmse:,.0f}', 'MAE ($)': f'{mae:,.0f}', 'R² Score': f'{r2:.4f}'})
    print(f'{name:<28} RMSE: ${rmse:>10,.0f}  |  MAE: ${mae:>9,.0f}  |  R²: {r2:.4f}')

results_df = pd.DataFrame(results)
print('\n')
results_df

In [ ]:
# ── Visual comparison of model performance ────────────────────────────────────
rmse_values = []
for name, model in models.items():
    if 'Regression' in name:
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(np.expm1(y_test), np.expm1(y_pred)))
    rmse_values.append(rmse)

colors = ['#e74c3c' if v == min(rmse_values) else 'steelblue' for v in rmse_values]

plt.figure(figsize=(10, 5))
bars = plt.bar(models.keys(), rmse_values, color=colors, edgecolor='white')
plt.title('Model Comparison: RMSE (Lower is Better)', fontsize=14, fontweight='bold')
plt.ylabel('RMSE ($)')
plt.xticks(rotation=15, ha='right')
for bar, val in zip(bars, rmse_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'${val:,.0f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Best model: Actual vs Predicted scatter ───────────────────────────────────
best_model = models['Gradient Boosting']
y_pred_best = np.expm1(best_model.predict(X_test))
y_test_actual = np.expm1(y_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual, y_pred_best, alpha=0.5, color='steelblue', s=30)
plt.plot([y_test_actual.min(), y_test_actual.max()],
         [y_test_actual.min(), y_test_actual.max()],
         'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Sale Price ($)')
plt.ylabel('Predicted Sale Price ($)')
plt.title('Gradient Boosting: Actual vs Predicted Prices', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature Importance & Business Insights

In [ ]:
# ── Top 20 features driving house price (Random Forest) ───────────────────────
rf_model = models['Random Forest']
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 8))
feature_importance.plot(kind='barh', color='steelblue', edgecolor='white')
plt.gca().invert_yaxis()
plt.title('Top 20 Most Important Features for Predicting House Price', fontsize=13, fontweight='bold')
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Business insight: Price premium per quality tier ─────────────────────────
quality_price = df.groupby('OverallQual')['SalePrice'].median()

plt.figure(figsize=(10, 5))
quality_price.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Median Sale Price by Overall Quality Rating', fontsize=13, fontweight='bold')
plt.xlabel('Overall Quality (1 = Very Poor, 10 = Very Excellent)')
plt.ylabel('Median Sale Price ($)')
plt.xticks(rotation=0)
for i, val in enumerate(quality_price):
    plt.text(i, val + 2000, f'${val/1000:.0f}K', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('quality_vs_price.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conclusions

### Model Performance Summary
| Model | Key Takeaway |
|---|---|
| Linear Regression | Baseline; struggles with non-linear relationships |
| Ridge / Lasso | Better than plain linear via regularization |
| Random Forest | Strong performance, robust to outliers |
| **Gradient Boosting** | **Best performer — lowest RMSE, highest R²** |

### Business Insights
1. **Overall Quality** is the single strongest predictor of sale price — a jump from quality 5→7 can add ~$50K in value.
2. **Total Square Footage** (including basement) matters more than above-ground area alone.
3. **Neighborhood** has a massive impact — the difference between the cheapest and most expensive neighborhoods exceeds $150K.
4. **Age & Remodeling** — newer or recently remodeled homes command a significant premium.
5. **Garage Area** is a surprisingly strong predictor, reflecting buyer preference in car-dependent markets.

### Next Steps
- Hyperparameter tuning with GridSearchCV
- Try XGBoost or LightGBM for even better performance
- Build an interactive Streamlit web app for price prediction
- Apply model to the Kaggle test set and submit for leaderboard scoring